# 🛍️ ShopEase Customer Support Chatbot
### ELIZA-Style Rule-Based Chatbot — Customer Service Domain

**Name:** Omamah Ajmal &nbsp;|&nbsp; **Roll No:** 42

---

## 📋 Introduction & Design

### Purpose
ShopEase is a **rule-based customer support chatbot** built for a fictional e-commerce store. Its purpose is to handle the most common customer enquiries — order tracking, returns, refunds, account issues, payment questions, and delivery times — without needing a human agent.

### Domain: Customer Service
The customer service domain was chosen because it has well-defined, predictable conversation patterns. Customers typically ask about a small set of topics (orders, returns, payments, account access), making it ideal for a regex pattern-matching approach.

### How It Works (Algorithm)
The chatbot follows the classic **ELIZA pattern-matching pipeline**:

1. **Input** — The user types a message.
2. **Pattern Matching** — The input is tested against an ordered list of `(regex, [responses])` tuples using `re.search()` with `IGNORECASE`.
3. **Pronoun Reflection** — Capture groups are passed through `reflect()`, which maps first-person words ("I", "my", "we") to second-person ("you", "your") so responses read naturally.
4. **Template Filling** — A response template is chosen randomly from the matched pattern's list. Capture group values replace `{0}`, `{1}` placeholders.
5. **Topic State** — A `state` dictionary tracks the last topic discussed (tracking / refund / return) so that a bare order number typed in the next turn is understood in context.
6. **Fallback** — If no pattern matches, a generic clarification response is returned.

### Pattern Groups
Patterns are organised into six thematic groups:
- `PATTERNS_GREETINGS` — Hellos, names, order-number follow-up
- `PATTERNS_ORDERS` — Order status, missing/delayed deliveries
- `PATTERNS_RETURNS` — Return requests, damaged/wrong items
- `PATTERNS_SHOPPING` — Promo codes, stock availability, cart help
- `PATTERNS_ACCOUNT` — Password reset, login issues, payment methods, address changes
- `PATTERNS_SENTIMENT` — Frustration, compliments, general problems, closing phrases

---

> **Run all cells top-to-bottom** — the interactive chat UI appears at the end of the last cell.


## 0. Install / Ensure Dependencies

In [ ]:
# Name: Omamah Ajmal | Roll No: 42
import sys

# Detect environment and install ipywidgets the right way
try:
    import ipywidgets
except ModuleNotFoundError:
    if sys.platform == 'emscripten':
        # JupyterLite / Pyodide — must use micropip, run via asyncio
        import asyncio, micropip
        asyncio.get_event_loop().run_until_complete(micropip.install('ipywidgets'))
        print('✅ ipywidgets installed via micropip.')
    else:
        # Colab / standard Jupyter — use pip
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
        print('✅ ipywidgets installed — please re-run this cell.')

# Enable widget manager for Colab
try:
    import google.colab
    IN_COLAB = True
    from google.colab import output
    output.enable_custom_widget_manager()
    print('✅ Colab detected — custom widget manager enabled.')
except ImportError:
    IN_COLAB = False
    env = 'JupyterLite' if sys.platform == 'emscripten' else 'Jupyter'
    print(f'✅ {env} environment detected.')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import re, random, html, datetime
print(f'✅ ipywidgets {widgets.__version__} ready.')


## 1. Store Constants
Define the store name, support email, and portal URLs used throughout responses.


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
STORE_NAME     = "ShopEase"
SUPPORT_EMAIL  = "support@shopease.example.com"
RETURNS_PORTAL = "shopease.example.com/returns"
TRACKING_URL   = "shopease.example.com/track"

## 2. Reflection Map
Maps first-person pronouns/verbs → second-person so the bot's echoed responses feel natural.
e.g. *"I haven't received my order"* → *"you haven't received your order"*


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
REFLECTIONS = {
    "i'm":       "you're",
    "i've":      "you've",
    "i'll":      "you'll",
    "i'd":       "you'd",
    "i":         "you",
    "me":        "you",
    "my":        "your",
    "mine":      "yours",
    "am":        "are",
    "was":       "were",
    "we're":     "you're",
    "we've":     "you've",
    "we'll":     "you'll",
    "we":        "you",
    "us":        "you",
    "our":       "your",
    "ours":      "yours",
    "myself":    "yourself",
    "ourselves": "yourselves",
}

def reflect(text: str) -> str:
    words = text.lower().split()
    result = []
    for word in words:
        clean = re.sub(r'[.,!?]+$', '', word)
        punct = word[len(clean):]
        result.append(REFLECTIONS.get(clean, clean) + punct)
    return ' '.join(result)

## 3. Pattern-Response Table

Each entry is a `(regex_pattern, [responses])` tuple.
- **Capture groups** in the regex map to `{0}`, `{1}` … in response templates.
- Multiple response strings per pattern ensure variety (one is chosen randomly).
- The sentinel string `"__ORDER_NUMBER_FOLLOWUP__"` triggers context-aware order handling.
- Patterns are tested **in order** — more specific patterns must come before general ones.


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_GREETINGS = [
    (
        r'^\s*#?(\d{3,})\s*$',
        ["__ORDER_NUMBER_FOLLOWUP__"]
    ),
    (
        r'^\s*(?:hello+|hi+|hey+|good\s+(?:morning|afternoon|evening))\b(.*)',
        [
            f"Hey there! Welcome to {STORE_NAME} support. Are you checking on an order, looking for a refund, or something else?",
            f"Hi! Thanks for reaching out to {STORE_NAME}. What can I help you with today?",
            f"Hello and welcome to {STORE_NAME}! How can I make your day better?",
        ]
    ),
    (
        r".*\bmy name(?:'?s| is)\s+([\w]+)",
        [
            f"Nice to meet you, {{0}}! I'm the {STORE_NAME} virtual assistant. What can I help you with?",
            f"Welcome, {{0}}! How can {STORE_NAME} help you today?",
        ]
    ),
]

In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_ORDERS = [
    (
        r'(.*\b(?:order|#)\b.*?#?\s*(\d{3,}).*)',
        [
            f"Thanks for sharing that! For order #{{1}}, the fastest way to get real-time status is to visit {TRACKING_URL} — you'll need the email used at checkout. If you're still stuck, email us at {SUPPORT_EMAIL} with your order number and we'll chase it up.",
            f"Got it — order #{{1}} noted. To track it live, head to {TRACKING_URL}. If something looks wrong, email {SUPPORT_EMAIL} and our team will investigate.",
            f"Thanks! For the latest status on order #{{1}}, check {TRACKING_URL}. If it's not showing, reach out to {SUPPORT_EMAIL} with your order number and they'll look into it.",
        ]
    ),
    (
        r"(i\b.*?\b(?:haven't|have\s+not|didn't|not\s+yet|still\s+waiting|never)\b.*?\b(?:received|arrived|come|shown\s+up|delivered|here|turned\s+up)\b.*)",
        [
            "I'm sorry to hear that. Could you share your order number? Once I have it, I can point you to the right tracking page or escalation route.",
            "That's frustrating, and I understand. Please share your order number and I'll direct you to the best way to get this resolved quickly.",
            f"Delayed or missing deliveries are stressful — sorry about that. Share your order number and I can guide you to the tracking portal or connect you with our support team at {SUPPORT_EMAIL}.",
        ]
    ),
    (
        r'(i\b.*?(?:still|not|never)\b.*?\b(?:order|package|parcel|delivery)\b.*)',
        [
            "I'm sorry you're having trouble. Could you share your order number so I can point you to the right next step?",
            "That doesn't sound right at all. Share your order number and I'll guide you on how to get it sorted as quickly as possible.",
        ]
    ),
    (
        r'.*\b(?:track|where|status|find|locate)\b.*\b(?:order|package|parcel|delivery|shipment)\b(.*)',
        [
            f"To track your order, head to {TRACKING_URL} — you'll need the email address used at checkout. If you have your order number handy I can help guide you further.",
            f"The quickest way is our tracking portal at {TRACKING_URL}. Do you have your order number? I can help point you in the right direction from there.",
        ]
    ),
]


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_RETURNS = [
    (
        r'.*\b(?:return|send\s+back|sending\s+back|give\s+back)\b\s+(.*)',
        [
            f"Of course — I can help you return {{0}}. Head to {RETURNS_PORTAL} for a free prepaid label, or share your order number and I'll start the process here.",
            f"Returning {{0}} is easy. Visit {RETURNS_PORTAL} or drop your order number here.",
            "Sure — returning {0} is straightforward. Could you share your order number so I can get that started?",
        ]
    ),
    (
        r'.*\b(?:return|send\s+back)\b(.*)',
        [
            f"Happy to help with a return! Visit {RETURNS_PORTAL} or share your order number here and I'll get it sorted.",
            f"Returns at {STORE_NAME} are easy — head to {RETURNS_PORTAL} for a free prepaid label.",
        ]
    ),
    (
        r'(?:i\b.*?\b)?((?:refund|money\s+back|reimburs\w+|paid\s+too\s+much).*)',
        [
            "I understand you're asking about a {0}. Refunds are processed within 3-5 business days. Could you share your order number so I can raise this?",
            "I can certainly help you with that {0}. I'll get that sorted — what's your order number and the reason for the refund?",
        ]
    ),
    (
        r'(i\b.*?\b(?:damaged|broken|faulty|defective|wrong\s+item|incorrect|not\s+what\s+i\s+ordered)\b.*)',
        [
            "I'm really sorry that {0}. That's completely unacceptable. Please share your order number and a photo if possible — I'll arrange a replacement or full refund immediately.",
            "I can see that {0}. This shouldn't happen. Give me your order number and I'll sort a replacement or refund right away.",
        ]
    ),
    (
        r'.*\b(damaged|broken|faulty|defective|wrong)\b.*\b(?:arrived|delivered|received|came)\b(.*)',
        [
            "I'm sorry to hear a {0} item arrived — that's not acceptable. Could you share your order number so I can arrange a replacement or refund?",
            "Receiving a {0} item is completely unacceptable. Please share your order number and I'll resolve this immediately.",
        ]
    ),
]

In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_SHOPPING = [
    (
        r'.*\b(?:promo(?:\s+code)?|discount|voucher|coupon|offer\s+code)\b(.*)',
        [
            f"{STORE_NAME} regularly runs promotions! Sign up at shopease.example.com for 10% off your next order. Do you have a specific code that isn't working?",
            "If your promo code isn't applying at checkout, check it hasn't expired and that your basket meets the minimum spend. Share the code here and I'll look into it!",
            "We do have discounts and offers available! Find all current deals at shopease.example.com/offers, or sign up to our newsletter for exclusive codes.",
        ]
    ),
    (
        r'.*\b(?:add|put|place)\b\s+(.*?)\s+\b(?:in|into|to)\b.*\b(cart|basket|bag)\b(.*)',
        [
            "Great choice! I can't directly modify your {1} from this chat, but you can easily add {0} by searching for it on shopease.example.com.",
            "I see you'd like to add {0} to your {1}. Just head to the product page on our site and click 'Add to {1}' to grab it!",
            "To add {0} to your {1}, visit shopease.example.com, find the product, and hit the 'Add to {1}' button. Is there anything else I can help with?",
        ]
    ),
    (
        r'.*\b(?:how\s+do\s+i|where\s+do\s+i|how\s+can\s+i)\b.*\b(?:add|put|place)\b.*\b(?:cart|basket|bag)\b(.*)',
        [
            "Easy! Go to the product page on shopease.example.com, choose your size or colour, and click the 'Add to Cart' button. It'll be in your basket instantly.",
            "To add items to your cart: search for the product on shopease.example.com, select any options like size or colour, then click 'Add to Cart'.",
        ]
    ),
    (
        r'.*\b(?:in\s+stock|available|back\s+in\s+stock|do\s+you\s+(?:have|sell|carry|stock))\b\s*(.*)',
        [
            "Stock levels change daily! Check the product page on shopease.example.com and click 'Notify Me' to get an email the moment it's back.",
            "For up-to-date stock info the product page is your best bet. Want me to flag this with our stock team?",
        ]
    ),
    (
        r'.*\b(?:looking\s+for|need|want\s+to\s+buy|shopping\s+for)\b\s+(\w[\w\s]*?)\b(?:\?|$|\.)',
        [
            "We carry a wide range of {0}! Browse shopease.example.com to see the full selection. Want me to flag a specific item for a stock alert?",
            "Great choice — we stock {0} across several brands. Head to shopease.example.com/search or I can flag the item for a 'back in stock' notification.",
        ]
    ),
]


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_ACCOUNT = [
    (
        r'.*\b(?:forgot(?:ten)?|reset|lost)\b.*\b(?:password|login|credentials)\b(.*)',
        [
            f"No problem — just click 'Forgot Password' on the {STORE_NAME} login page and we'll email you a reset link within a minute.",
            f"Easy fix! Hit 'Forgot Password' on the login page. If the email doesn't arrive within 5 minutes, check your spam folder or email us at {SUPPORT_EMAIL}.",
        ]
    ),
    (
        r"(i\b.*?\b(?:can't|cannot|unable\s+to|trouble|problem|issue)\b.*?\b(?:log\s*in|login|sign\s*in|access\s+my\s+account)\b.*)",
        [
            f"I'm sorry that {{0}}. Try the 'Forgot Password' link on the login page. If that doesn't work, email {SUPPORT_EMAIL} with your registered address and we'll reset it manually.",
            "I can see that {0}. Account access issues are usually fixed with a quick password reset — have you tried 'Forgot Password'?",
        ]
    ),
    (
        r'.*\b(?:accept|take|pay\s+with)\b\s+(paypal|credit\s+card|debit\s+card|apple\s+pay|google\s+pay|klarna|afterpay|cash)\b(.*)',
        [
            "Yes, we accept {0} at checkout! You'll see the option when you are ready to complete your order.",
            "You can definitely use {0} for your purchase. We also accept most other major payment methods.",
        ]
    ),
    (
        r'.*\b(?:payment\s+methods?|how\s+(?:can|do)\s+(?:i|we)\s+pay|options\s+to\s+pay)\b(.*)',
        [
            "We accept all major credit and debit cards, PayPal, Apple Pay, and Google Pay. You'll see all available payment methods at the checkout page.",
            "You can pay using Visa, Mastercard, American Express, PayPal, and Apple Pay. Is there a specific method you were hoping to use?",
        ]
    ),
    (
        r'(i\b.*?\b(?:change|update|edit|modify)\b.*?\b(?:address|email|phone|number|payment|card)\b.*)',
        [
            f"I understand that {{0}}. You can make that change in 'Account Settings' once logged in. If you're stuck, email {SUPPORT_EMAIL} and we'll update it for you.",
            "I can see that {0}. Head to Account Settings — it should only take a moment. Anything stopping you from getting in?",
        ]
    ),
    (
        r'.*\b(?:how\s+long|when\s+will|when\s+can|estimated|expected|how\s+soon)\b.*\b(?:deliver|shipping|arrive|receipt|receive|dispatch|sent|get\s+my)\b(.*)',
        [
            f"Standard delivery at {STORE_NAME} takes 3-5 business days; express is 1-2 days. You'll get a tracking email as soon as your order is dispatched.",
            "Delivery times depend on your location and chosen shipping method. Do you have an order number I can check the dispatch status on?",
        ]
    ),
]

In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS_SENTIMENT = [
    (
        r'(i\b.*?\b(?:frustrated|unhappy|angry|disappointed|upset|annoyed|furious|not\s+happy)\b.*)',
        [
            f"I'm truly sorry to hear {{0}}. That's not the {STORE_NAME} experience we want for you — please tell me more and I'll do everything I can to put it right.",
            "I understand {0}. Your feelings are completely valid and I want to fix this. Can you give me a bit more detail?",
            "I hear that {0}. I apologise — please share your order number or describe the issue and I'll escalate it immediately.",
        ]
    ),
    (
        r'(.*\b(?:terrible|awful|horrible|rubbish|poor|disgusting|useless|dreadful)\b.*)',
        [
            f"I'm really sorry you feel that {{0}}. That's not the standard {STORE_NAME} stands for — could you tell me what happened so I can escalate it?",
            "I hear you that {0}. I sincerely apologise — please describe what went wrong and I'll make sure it reaches the right team.",
        ]
    ),
    (
        r'(i\b.*?\b(?:problem|issue|trouble|concern)\b.*)',
        [
            "I can see that {0}. I'm sorry to hear that — could you give me a bit more detail and your order number if relevant?",
            "I understand that {0}. Let's sort this out together. What exactly is going wrong?",
        ]
    ),
    (
        r'(i\b.*?\b(?:love|amazing|excellent|fantastic|brilliant|happy\s+with|impressed|wonderful)\b.*)',
        [
            f"That's so wonderful to hear that {{0}}! We'll pass your kind words on to the team. Thanks for shopping with {STORE_NAME}! 🎉",
            f"It's brilliant to hear that {{0}}. Feedback like this really makes our day at {STORE_NAME}!",
        ]
    ),
    (
        r'.*\b(?:can|could)\b\s+(?:you|u)\s+(.*)',
        [
            "I think I can {0}. I'm an automated assistant, but I'll do my best!",
            "I'm here to help with orders and returns. I can certainly try to {0}.",
            f"I can help you with most {STORE_NAME} tasks. Ask me specifically about your order!",
        ]
    ),
    (
        r'^\s*why\b\s+(.*)',
        [
            "That's a good question. Why do you think {0}?",
            "I'm not sure why {0}, but I can look into it if you have an order number.",
            "The reason {0} usually depends on stock or shipping delays. Can I check your order?",
        ]
    ),
    (
        r'.*\b(?:mean|meant)\b\s+(.*)',
        [
            "Oh, I see! You meant {0}. Thanks for clarifying.",
            "Understood. Since you meant {0}, let's try to sort that out.",
        ]
    ),
    (
        r'.*\b(?:bye|goodbye|see\s+you|cheers|thank?\s*you|thanks|thx|ty)\b(.*)',
        [
            f"Thanks for reaching out to {STORE_NAME}! Have a brilliant day. 👋",
            "It was a pleasure helping you today. Happy shopping!",
            f"Goodbye! If you ever need anything else, {STORE_NAME} support is always here.",
        ]
    ),
]

In [ ]:
# Name: Omamah Ajmal | Roll No: 42
PATTERNS = (
    PATTERNS_GREETINGS
    + PATTERNS_ORDERS
    + PATTERNS_RETURNS
    + PATTERNS_SHOPPING
    + PATTERNS_ACCOUNT
    + PATTERNS_SENTIMENT
)
print(f"✅ Total patterns loaded: {len(PATTERNS)}")

## 4. Core Bot Logic

`get_bot_response(user_input)` iterates through `PATTERNS` and returns the first match.
It handles:
- **Order-number follow-up** — if the user just types a number after asking about tracking/refund/return, the `state` dict provides context.
- **Group reflection** — captured text is passed through `reflect()` before being inserted into the template.
- **Topic tracking** — sets `state["last_topic"]` when a response asks for an order number.
- **Fallback** — returns a clarifying question if nothing matches.


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
state = {"last_topic": ""}

def get_bot_response(user_input: str) -> str:
    text = user_input.strip()
    for pattern, responses in PATTERNS:
        match = re.search(pattern, text, re.IGNORECASE)
        if not match:
            continue
        if responses[0] == "__ORDER_NUMBER_FOLLOWUP__":
            order_num = match.group(1)
            topic = state["last_topic"]
            if topic == "tracking":
                reply = random.choice([
                    f"Thanks! For order #{order_num}, visit {TRACKING_URL} with your checkout email for live status. If it looks off, email {SUPPORT_EMAIL} quoting that order number.",
                    f"Got it — order #{order_num}. Head to {TRACKING_URL} to see the latest delivery update. Still stuck? Our team at {SUPPORT_EMAIL} can investigate directly.",
                ])
            elif topic == "refund":
                reply = random.choice([
                    f"Thanks for order #{order_num}. To request a refund, please email {SUPPORT_EMAIL} with that order number and the reason — our team typically responds within 1 business day.",
                    f"Noted — order #{order_num}. Refund requests are handled by our support team at {SUPPORT_EMAIL}. Quote your order number and they'll get it sorted for you.",
                ])
            elif topic == "return":
                reply = random.choice([
                    f"Thanks — order #{order_num} noted. Head to {RETURNS_PORTAL} to generate your free prepaid return label. If you run into any issues, email {SUPPORT_EMAIL}.",
                    f"Got it! Visit {RETURNS_PORTAL} and enter order #{order_num} to start your return and get a prepaid label emailed to you.",
                ])
            else:
                reply = random.choice([
                    f"Thanks — order #{order_num} noted. For the fastest help, email {SUPPORT_EMAIL} with that number and a brief description of your issue.",
                    f"Got it — order #{order_num}. Our support team at {SUPPORT_EMAIL} will be best placed to help with the specifics.",
                ])
            state["last_topic"] = ""
            return reply

        groups = [reflect(g.strip()) if g else "" for g in match.groups()]
        template = random.choice(responses)
        reply = template
        for i, val in enumerate(groups):
            reply = reply.replace("{" + str(i) + "}", val)

        if "order number" in reply.lower():
            lower = text.lower()
            if any(w in lower for w in ["track", "where", "status", "arrived", "received", "delivery"]):
                state["last_topic"] = "tracking"
            elif any(w in lower for w in ["refund", "money back", "reimburs"]):
                state["last_topic"] = "refund"
            elif any(w in lower for w in ["return", "send back"]):
                state["last_topic"] = "return"
            else:
                state["last_topic"] = "general"
        return reply

    if "?" in text:
        return random.choice([
            "That's an interesting question. I'm not sure I have the exact answer, but have you checked our FAQ at shopease.example.com/help?",
            "I'm not completely sure about that. Could you try asking it differently or share an order number?",
        ])
    return random.choice([
        "I'm not quite sure I understood that. Could you rephrase it or give me a bit more detail?",
        "I'm still learning and might have missed your point! Could you describe the issue differently?",
        f"I want to help, but I'm having trouble understanding. You can also email us at {SUPPORT_EMAIL}.",
    ])

print("✅ Bot logic ready.")


## 6. 🧪 Automated Tests

Run this cell to verify the bot responds sensibly to a range of inputs covering every pattern group.


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
test_cases = [
    # Greetings
    ("Hi there!",                          "greeting"),
    ("My name is Omamah",                   "name recognition"),
    # Orders & Tracking
    ("Where is my order #45231?",           "order with number"),
    ("I haven't received my package yet",   "missing delivery"),
    ("Can you track my shipment?",          "tracking request"),
    # Order number follow-up (context-aware)
    ("I haven't received my order",        "sets tracking topic"),
    ("99887",                              "bare order number -> tracking context"),
    # Returns & Refunds
    ("I want to return my shoes",          "return with item"),
    ("I'd like a refund please",           "refund request"),
    ("My item arrived damaged",            "damaged item"),
    ("I received the wrong item",          "wrong item"),
    # Shopping
    ("Do you have a discount code?",       "promo code"),
    ("How do I add items to my cart?",     "cart help"),
    ("Is the blue jacket in stock?",       "stock check"),
    # Account & Payments
    ("I forgot my password",               "password reset"),
    ("I can't log in to my account",      "login issue"),
    ("Do you accept PayPal?",              "payment method"),
    ("How long does delivery take?",       "delivery time"),
    ("I want to change my address",        "account update"),
    # Sentiment
    ("I am really frustrated",             "negative sentiment"),
    ("I love shopping here!",              "positive sentiment"),
    ("I have a problem with my order",     "general problem"),
    # Closing
    ("Thanks, bye!",                       "closing"),
    # Fallback
    ("Blah blah xyz???",                   "fallback with question mark"),
    ("asjdhaksjdhaksjdh",                  "pure fallback"),
]

state['last_topic'] = ''
print(f"{'Input':<42} {'Category':<35} Response")
print("-" * 120)
for user_msg, category in test_cases:
    response = get_bot_response(user_msg)
    print(f"{user_msg:<42} [{category:<33}] {response[:70]}{'...' if len(response)>70 else ''}")


## 5. 🎨 Interactive Chat UI

Built with `ipywidgets` + `IPython.display`. Renders a fully styled chat interface inside the notebook.
- **Hidden ipywidgets** handle Python callbacks (send, reset).
- **Visible HTML/CSS/JS** provides the chat window, message bubbles, timestamps, and typing indicator.
- The HTML `<input>` bridges to the hidden widget via JavaScript so only one input bar is visible.


In [ ]:
# Name: Omamah Ajmal | Roll No: 42
# ── Styles ──────────────────────────────────────────────────────────────────
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&display=swap');

:root {
  --teal:      #00897b;
  --teal-dark: #00695c;
  --teal-pale: #e0f2f1;
  --user-bg:   #1a237e;
  --user-text: #ffffff;
  --bot-bg:    #ffffff;
  --bot-text:  #1c1c1e;
  --surface:   #f7f8fc;
  --border:    #e3e8ef;
  --muted:     #8a95a3;
  --radius:    18px;
  --font:      'Plus Jakarta Sans', sans-serif;
}

#shopease-chat-root * { box-sizing: border-box; font-family: var(--font); }

#shopease-chat-root {
  max-width: 680px;
  margin: 16px auto;
  border-radius: 20px;
  overflow: hidden;
  box-shadow: 0 8px 40px rgba(0,0,0,.14);
  border: 1px solid var(--border);
  background: var(--surface);
}

/* Header */
.se-header {
  background: linear-gradient(135deg, var(--teal) 0%, #26a69a 100%);
  padding: 18px 24px;
  display: flex;
  align-items: center;
  gap: 14px;
}
.se-avatar {
  width: 46px; height: 46px;
  border-radius: 50%;
  background: rgba(255,255,255,.25);
  display: flex; align-items: center; justify-content: center;
  font-size: 22px;
  flex-shrink: 0;
}
.se-header-info h3 { margin: 0; color: #fff; font-size: 16px; font-weight: 700; }
.se-header-info p  { margin: 2px 0 0; color: rgba(255,255,255,.8); font-size: 12px; }
.se-status-dot {
  width: 8px; height: 8px;
  background: #69f0ae;
  border-radius: 50%;
  display: inline-block;
  margin-right: 4px;
  animation: pulse 2s infinite;
}
@keyframes pulse {
  0%,100% { opacity: 1; } 50% { opacity: .4; }
}

/* Message window */
.se-messages {
  height: 420px;
  overflow-y: auto;
  padding: 20px 18px;
  display: flex;
  flex-direction: column;
  gap: 10px;
  background: var(--surface);
  scroll-behavior: smooth;
}
.se-messages::-webkit-scrollbar { width: 4px; }
.se-messages::-webkit-scrollbar-thumb { background: var(--border); border-radius: 4px; }

/* Bubbles */
.se-row { display: flex; align-items: flex-end; gap: 8px; }
.se-row.user { flex-direction: row-reverse; }
.se-mini-avatar {
  width: 30px; height: 30px;
  border-radius: 50%;
  background: var(--teal-pale);
  display: flex; align-items: center; justify-content: center;
  font-size: 14px;
  flex-shrink: 0;
}
.se-bubble-wrap { display: flex; flex-direction: column; max-width: 75%; }
.se-row.user .se-bubble-wrap { align-items: flex-end; }
.se-bubble {
  padding: 11px 15px;
  border-radius: var(--radius);
  font-size: 14px;
  line-height: 1.55;
  word-wrap: break-word;
  position: relative;
  animation: fadeUp .25s ease;
}
@keyframes fadeUp {
  from { opacity: 0; transform: translateY(8px); }
  to   { opacity: 1; transform: translateY(0); }
}
.bot .se-bubble  { background: var(--bot-bg);  color: var(--bot-text); box-shadow: 0 2px 8px rgba(0,0,0,.07); border-bottom-left-radius: 4px; }
.user .se-bubble { background: var(--user-bg); color: var(--user-text); border-bottom-right-radius: 4px; }
.se-time { font-size: 10px; color: var(--muted); margin-top: 4px; }

/* Typing indicator */
.se-typing { display: flex; gap: 5px; align-items: center; padding: 12px 16px; }
.se-dot {
  width: 7px; height: 7px;
  border-radius: 50%;
  background: var(--teal);
  animation: bounce 1.2s infinite;
}
.se-dot:nth-child(2) { animation-delay: .2s; }
.se-dot:nth-child(3) { animation-delay: .4s; }
@keyframes bounce {
  0%,80%,100% { transform: translateY(0); }
  40%          { transform: translateY(-6px); }
}


/* Divider */
.se-divider {
  height: 1px;
  background: var(--border);
  margin: 0;
}

/* Footer / input area */
.se-footer {
  background: #fff;
  padding: 12px 18px 16px;
  display: flex;
  align-items: center;
  gap: 10px;
}
.se-footer input[type=text] {
  flex: 1;
  border: 1.5px solid var(--border);
  border-radius: 24px;
  padding: 10px 18px;
  font-size: 14px;
  font-family: var(--font);
  outline: none;
  transition: border-color .2s;
  color: var(--bot-text);
  background: var(--surface);
}
.se-footer input[type=text]:focus { border-color: var(--teal); }
.se-send-btn {
  width: 42px; height: 42px;
  background: var(--teal);
  border: none;
  border-radius: 50%;
  cursor: pointer;
  display: flex; align-items: center; justify-content: center;
  flex-shrink: 0;
  transition: background .2s, transform .1s;
}
.se-send-btn:hover  { background: var(--teal-dark); }
.se-send-btn:active { transform: scale(.93); }
.se-send-btn svg { fill: #fff; }
.se-reset-btn {
  background: none;
  border: 1.5px solid var(--border);
  border-radius: 50%;
  width: 38px; height: 38px;
  cursor: pointer;
  color: var(--muted);
  font-size: 17px;
  display: flex; align-items: center; justify-content: center;
  transition: border-color .2s, color .2s;
}
.se-reset-btn:hover { border-color: var(--teal); color: var(--teal); }

/* System message */
.se-system {
  text-align: center;
  font-size: 11px;
  color: var(--muted);
  padding: 4px 0;
  font-style: italic;
}
</style>
"""

# ── State ───────────────────────────────────────────────────────────────────
chat_messages = []
state["last_topic"] = ""


def now_str():
    return datetime.datetime.now().strftime("%H:%M")

# ── HTML rendering ───────────────────────────────────────────────────────────
def render_bubble(role, text, time):
    safe = html.escape(text).replace("\n", "<br>")
    if role == "bot":
        return (
            '<div class="se-row bot">'
            '<div class="se-mini-avatar">🛍️</div>'
            '<div class="se-bubble-wrap">'
            f'<div class="se-bubble">{safe}</div>'
            f'<span class="se-time">{time}</span>'
            '</div></div>'
        )
    return (
        '<div class="se-row user">'
        '<div class="se-mini-avatar">👤</div>'
        '<div class="se-bubble-wrap">'
        f'<div class="se-bubble">{safe}</div>'
        f'<span class="se-time">{time}</span>'
        '</div></div>'
    )

def build_html(show_typing=False, system_msg=None):
    bubbles = ""
    for m in chat_messages:
        bubbles += render_bubble(m["role"], m["text"], m["time"])
    if show_typing:
        bubbles += (
            '<div class="se-row bot">'
            '<div class="se-mini-avatar">🛍️</div>'
            '<div class="se-bubble bot"><div class="se-typing">'
            '<div class="se-dot"></div><div class="se-dot"></div><div class="se-dot"></div>'
            '</div></div></div>'
        )
    if system_msg:
        bubbles += f'<div class="se-system">{html.escape(system_msg)}</div>'

    chips = ""
    scroll_js = """
(function(){
  // Auto-scroll to bottom
  function scrollBottom(){
    var m = document.getElementById('se-msgs');
    if(m) m.scrollTop = m.scrollHeight;
  }
  scrollBottom();
  setTimeout(scrollBottom, 200);

  // Bind Enter key on the visible input (guard against double-binding)
  var inp = document.getElementById('se-input');
  if(inp && !inp._seBound){
    inp._seBound = true;
    inp.addEventListener('keydown', function(e){
      if(e.key === 'Enter'){ e.preventDefault(); seSend(); }
    });
  }
  if(inp) setTimeout(function(){ inp.focus(); }, 150);
})();

function seGetHiddenInput(){
  var all = document.querySelectorAll('input[type=text]');
  for(var i = 0; i < all.length; i++){
    if(all[i].id !== 'se-input') return all[i];
  }
  return null;
}

function seGetHiddenBtn(){
  var btns = document.querySelectorAll('button');
  for(var i = 0; i < btns.length; i++){
    var t = btns[i].textContent.trim();
    if(t === 'Send ➤' || t === 'Send') return btns[i];
  }
  return null;
}

function seFireValue(inp, val){
  try{
    var setter = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype, 'value').set;
    setter.call(inp, val);
  }catch(e){ inp.value = val; }
  inp.dispatchEvent(new Event('input',  {bubbles: true}));
  inp.dispatchEvent(new Event('change', {bubbles: true}));
}

function seSend(){
  var seInp = document.getElementById('se-input');
  if(!seInp || !seInp.value.trim()) return;
  var val = seInp.value;
  seInp.value = '';
  var hiddenInp = seGetHiddenInput();
  if(hiddenInp){
    seFireValue(hiddenInp, val);
    setTimeout(function(){
      var btn = seGetHiddenBtn();
      if(btn){ btn.click(); }
      else{
        hiddenInp.dispatchEvent(new KeyboardEvent('keydown', {key:'Enter', bubbles:true}));
      }
    }, 80);
  }
}

function seSetAndSend(txt){
  var seInp = document.getElementById('se-input');
  if(seInp) seInp.value = txt;
  seSend();
}

function seReset(){
  var btns = document.querySelectorAll('button');
  for(var i = 0; i < btns.length; i++){
    if(btns[i].textContent.trim().startsWith('↺')){ btns[i].click(); break; }
  }
}
"""

    body = (
        '<div id="shopease-chat-root">'
        '<div class="se-header">'
        '<div class="se-avatar">🛍️</div>'
        '<div class="se-header-info">'
        '<h3>ShopEase Support</h3>'
        '<p><span class="se-status-dot"></span>Online &nbsp;·&nbsp; Virtual Assistant</p>'
        '</div></div>'
        f'<div class="se-messages" id="se-msgs">{bubbles}</div>'
        '<div class="se-divider"></div>'
        '<div class="se-footer">'
        '<button class="se-reset-btn" title="New conversation" onclick="seReset()">↺</button>'
        '<input id="se-input" type="text" placeholder="Type your message…" />'
        '<button class="se-send-btn" onclick="seSend()">'
        '<svg width="18" height="18" viewBox="0 0 24 24">'
        '<path d="M2 21l21-9L2 3v7l15 2-15 2z"/>'
        '</svg></button>'
        '</div>'
        '</div>'
        f'<script>{scroll_js}</script>'
    )
    return body

# ── Hidden ipywidgets (Python callbacks) ──────────────────────────────────
output_area = widgets.Output()
txt_input   = widgets.Text(
    placeholder="",
    layout=widgets.Layout(display='none')
)
btn_send    = widgets.Button(
    description="Send ➤",
    button_style="success",
    layout=widgets.Layout(display='none')
)
btn_reset   = widgets.Button(
    description="↺ New Chat",
    layout=widgets.Layout(display='none')
)

# JS run after every refresh — re-attaches Enter key & scrolls to bottom

def refresh(show_typing=False, system_msg=None):
    with output_area:
        clear_output(wait=True)
        display(HTML(CSS + build_html(show_typing, system_msg)))


def add_message(role, text):
    chat_messages.append({"role": role, "text": text, "time": now_str()})

def do_send(user_text):
    if not user_text.strip():
        return
    add_message("user", user_text)
    refresh(show_typing=True)
    import time; time.sleep(0.55)
    response = get_bot_response(user_text)
    add_message("bot", response)
    refresh()

def on_send(_):
    text = txt_input.value
    txt_input.value = ""
    do_send(text)

def on_enter(widget):
    text = widget.value
    widget.value = ""
    do_send(text)


def on_reset(_):
    chat_messages.clear()
    state["last_topic"] = ""
    add_message("bot", f"Hi! Welcome to {STORE_NAME} support. How can I help you today? 😊")
    refresh(system_msg="— New conversation started —")

btn_send.on_click(on_send)
btn_reset.on_click(on_reset)
# Enter is handled entirely via the HTML input keydown -> seSend() -> btn_send.click()

# ── Initial load ─────────────────────────────────────────────────────────────
add_message("bot", f"Hi! Welcome to {STORE_NAME} support. 👋\nI can help with orders, returns, refunds, account issues, and more. How can I help you today?")

# Show only the output area; hidden widgets remain functional as Python callbacks
display(output_area, txt_input, btn_send, btn_reset)
refresh()


---

## 7. 📝 Reflection

### Challenges Faced

**1. Regex Ordering & Specificity**  
The most difficult part was ordering patterns correctly. A general pattern like `.*\b(problem|issue)\b.*` would match inputs that should have been caught by more specific patterns (e.g. damaged items or login issues). Patterns had to be arranged from most specific to most general within each group, and tested extensively to avoid mis-fires.

**2. Pronoun Reflection Edge Cases**  
The `reflect()` function works word-by-word, which means contractions like *"I'm"* or *"I've"* needed explicit entries. Without them, the reflection would produce unnatural output like *"you I've"* instead of *"you've"*.

**3. Context Across Turns**  
ELIZA-style bots are stateless by nature. Adding the `state["last_topic"]` dictionary to handle order-number follow-ups (so that a bare number like `45231` is understood as a tracking request in context) required careful state management and resetting after each use.

**4. Sounding Natural Without Over-Promising**  
Early versions of the bot said things like *"I'm pulling up your order right now"*, which sounds like the bot has real system access. Responses were rewritten to honestly direct users to tracking portals and support emails while still sounding helpful and empathetic.

**5. Notebook UI Integration**  
Embedding a polished chat UI inside a Jupyter/Colab notebook required bridging HTML/CSS/JS with Python via `ipywidgets`. A key challenge was hiding the default widget input bar while keeping the Python callbacks functional, and ensuring compatibility across Colab, Jupyter, and JupyterLite.

### How the Domain Influenced Responses

Choosing **Customer Service** as the domain shaped every design decision:
- Responses needed to be **professional and empathetic**, especially for complaints and damaged items.
- The bot had to **acknowledge limitations honestly** (it cannot actually look up orders) while still being useful by directing users to the right resources.
- Common e-commerce vocabulary (order numbers, returns portals, prepaid labels, tracking URLs) was embedded directly into response templates.
- Sentiment patterns were important — a frustrated customer needs to feel heard before being given instructions.
- Multiple response variants per pattern reduce repetitiveness, which is critical in a customer-facing context.

Overall, the ELIZA approach proved well-suited to customer service because the domain has clearly bounded, predictable conversation flows — unlike open-ended domains such as mental health or relationship advice where responses require much more nuance.
